# PointTransformer V2 — Train / Validate / Test (NO Open3D)

Colab-safe version: Open3D is used elsewhere only for normal-estimation and
visualization — neither is needed to *train* PTv2. Normals here are computed
with plain NumPy + scikit-learn (PCA on local neighbourhoods) instead.

**Install (Colab — run the SETUP cell below once, then `Runtime → Restart runtime`,
then run everything from the top again).** Colab currently ships torch 2.6.0+cu124,
whose torch-cluster/torch-scatter wheel page is broken (403) — the setup cell
therefore pins torch 2.5.1+cu121, a combination whose CUDA wheels are verified to
exist.


In [ ]:
# ================================================================ COLAB SETUP (run ONCE, then Runtime -> Restart runtime)
# Colab's default torch 2.6.0+cu124 has NO working torch-cluster/torch-scatter CUDA
# wheels (the data.pyg.org page for that combo is broken). We pin torch 2.5.1+cu121,
# for which CUDA wheels exist, so PTv2 attention actually runs on the GPU.
import importlib.util, torch

need_install = True
try:
    if torch.__version__.startswith("2.5.1") and (torch.version.cuda or "").startswith("12.1"):
        from torch_cluster import knn as _k          # noqa
        x = torch.randn(8, 3, device="cuda") if torch.cuda.is_available() else None
        if x is not None:
            b = torch.zeros(8, dtype=torch.long, device="cuda")
            _k(x, x, 4, b, b)                        # CUDA smoke test
        need_install = False
        print("environment already OK -- skip to the next cell")
except Exception:
    pass

if need_install:
    print("installing pinned torch 2.5.1+cu121 + matching torch-scatter/torch-cluster ...")
    import subprocess
    subprocess.run("pip uninstall -y torch torchvision torchaudio torch-scatter torch-cluster",
                   shell=True)
    subprocess.run("pip install torch==2.5.1 torchvision torchaudio "
                   "--index-url https://download.pytorch.org/whl/cu121", shell=True)
    subprocess.run("pip install torch-scatter torch-cluster "
                   "-f https://data.pyg.org/whl/torch-2.5.1+cu121.html", shell=True)
    subprocess.run("pip install laspy tqdm", shell=True)
    print("\n" + "=" * 60)
    print(">>> NOW: Runtime -> Restart runtime, then run from the top. <<<")
    print("=" * 60)

In [ ]:
# ================================================================ IMPORTS (no open3d)
import os, glob, time, copy, random, logging, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix
import laspy
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s")
log = logging.getLogger("ptv2")

try:
    from torch_cluster import knn as tc_knn
    from torch_scatter import scatter_softmax, scatter_add, scatter_max, scatter_mean
    HAS_SCATTER = True
except Exception:
    HAS_SCATTER = False
    raise RuntimeError("torch_cluster / torch_scatter are required for PTv2. "
                      "Install matching wheels from https://data.pyg.org/whl/")

def _pick_device():
    """CUDA থাকলেও torch_cluster-এর CUDA kernel আসলে কাজ করে কিনা যাচাই করে;
    mismatched wheel হলে crash না করে CPU-তে fallback করে।"""
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        x = torch.randn(10, 3, device="cuda")
        b = torch.zeros(10, dtype=torch.long, device="cuda")
        tc_knn(x, x, 4, b, b)
        return torch.device("cuda")
    except Exception as e:
        log.warning("CUDA detected but torch_cluster can't use it (%s) -> CPU "
                   "fallback. Run the SETUP cell + Restart runtime to fix.", e)
        return torch.device("cpu")

DEVICE = _pick_device()
log.info("PyTorch %s | device: %s", torch.__version__, DEVICE)

In [ ]:
# ================================================================ CONFIG
CONFIG = {
    "train_dir": "data/train",      # labelled .las/.laz/.xyz/.pts/.txt files
    "test_dir":  "data/test",       # final inference only (no metrics if unlabeled)
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "num_points": 4096,             # points per training chunk (sphere-crop)
    "chunks_per_cloud": 4,
    "batch_size": 8,
    "grad_accum": 1,
    "epochs": 100,
    "patience": 30,                 # early stop if val mIoU doesn't improve
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "amp": True,
    "k_neighbors": 16,               # PTv2 attention neighbourhood size
    "normal_k": 16,                  # neighbourhood size for normal estimation
    "num_classes": 2,               # 0=environment, 1=wood powder (auto-checked below)
    "seed": 42,
    "checkpoint_path": "checkpoints/ptv2_no_o3d_best.pth",
    # 7-ch feature cache: normals are expensive to compute (minutes per big file),
    # so we persist them to disk once. Point this at Google Drive to survive
    # Colab session resets, e.g. "/content/drive/MyDrive/features_cache.pkl".
    "feature_cache_file": "features_cache.pkl",
}
os.makedirs(os.path.dirname(CONFIG["checkpoint_path"]), exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(CONFIG["seed"])

## Data loading & 7-channel features (Open3D-free)

Each point gets **7 channels**: normalized `(x, y, z)` + `height-above-floor` +
`(nx, ny, nz)` surface normal. Normals are estimated by PCA on each point's
k-nearest neighbours (`scikit-learn NearestNeighbors` finds the neighbours,
NumPy does the eigendecomposition) — mathematically identical to what Open3D's
`estimate_normals` does internally, just without the Open3D dependency.

In [ ]:
# ================================================================ LOADER (no open3d)
def load_pointcloud(path):
    """(points[N,3] float64, labels[N] int64 or None). Supports .las/.laz/.xyz/.pts/.txt"""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".las", ".laz"):
        las = laspy.read(path)
        pts = np.column_stack((np.asarray(las.x), np.asarray(las.y),
                               np.asarray(las.z))).astype(np.float64)
        lbl = None
        if "classification" in las.point_format.dimension_names:
            lbl = np.asarray(las.classification, dtype=np.int64)
        return pts, lbl
    # .xyz / .pts / .txt
    for kw in (dict(), dict(delimiter=","), dict(skiprows=1),
               dict(delimiter=",", skiprows=1)):
        try:
            arr = np.loadtxt(path, **kw); break
        except ValueError:
            arr = None
    if arr is None:
        arr = np.genfromtxt(path, delimiter=",", skip_header=1)
    arr = np.asarray(arr, np.float64)
    lbl = arr[:, 3].astype(np.int64) if arr.shape[1] >= 4 else None
    return arr[:, :3], lbl

def list_files(folder):
    out = []
    for e in (".las", ".laz", ".xyz", ".pts", ".txt"):
        out += glob.glob(os.path.join(folder, "*" + e))
    return sorted(out)

def estimate_normals_no_o3d(pts, k=16):
    """PCA-based normal estimation, no Open3D: for each point, take its k nearest
    neighbours, build the local covariance matrix, and use the eigenvector of the
    SMALLEST eigenvalue as the normal (the flattest local direction)."""
    n = len(pts)
    k = min(k, n - 1) if n > 1 else 1
    nbrs = NearestNeighbors(n_neighbors=k, algorithm="kd_tree").fit(pts)
    _, idx = nbrs.kneighbors(pts)             # (N,k)
    neigh = pts[idx]                          # (N,k,3)
    mean = neigh.mean(1, keepdims=True)
    cov = np.einsum("nki,nkj->nij", neigh - mean, neigh - mean) / k
    eigvals, eigvecs = np.linalg.eigh(cov)    # ascending eigenvalues
    normals = eigvecs[:, :, 0]                # eigenvector of smallest eigenvalue
    flip = normals[:, 2] < 0                  # orient roughly "upward" (+z)
    normals[flip] *= -1
    return normals.astype(np.float32)

def build_features(pts, normal_k):
    """xyz (normalized) + height-above-floor + normals -> (N,7) float32."""
    center = pts.mean(0, keepdims=True)
    scale = max(np.linalg.norm(pts - center, axis=1).max(), 1e-9)
    norm_xyz = ((pts - center) / scale).astype(np.float32)
    height = ((pts[:, 2] - pts[:, 2].min())
             / max(pts[:, 2].max() - pts[:, 2].min(), 1e-6)).astype(np.float32)
    normals = estimate_normals_no_o3d(pts, k=normal_k)
    return np.column_stack([norm_xyz, height[:, None], normals]).astype(np.float32)

In [ ]:
# ================================================================ SPLITS & CACHE
TRAIN_FILES_ALL = list_files(CONFIG["train_dir"])
TEST_FILES = list_files(CONFIG["test_dir"])
assert TRAIN_FILES_ALL, f"No files found in {CONFIG['train_dir']}"
log.info("train pool: %d files | final-inference: %d files",
         len(TRAIN_FILES_ALL), len(TEST_FILES))

rng = np.random.RandomState(CONFIG["seed"])
perm = rng.permutation(len(TRAIN_FILES_ALL))
n_val = max(1, int(len(TRAIN_FILES_ALL) * CONFIG["val_ratio"]))
n_test = max(1, int(len(TRAIN_FILES_ALL) * CONFIG["test_ratio"]))
SPLIT = {
    "train": [TRAIN_FILES_ALL[i] for i in perm[n_val + n_test:]],
    "val":   [TRAIN_FILES_ALL[i] for i in perm[:n_val]],
    "test":  [TRAIN_FILES_ALL[i] for i in perm[n_val:n_val + n_test]],   # held-out
}
log.info("split -> train %d | val %d | held-out test %d",
         len(SPLIT["train"]), len(SPLIT["val"]), len(SPLIT["test"]))

class CloudCache:
    """In-RAM cache: raw points, 7-ch features, labels -- computed once per file.
    CRITICAL: limit must cover EVERY train+val file, otherwise files evicted
    during an epoch get their expensive normals recomputed EVERY epoch and
    training crawls (GPU sits idle while sklearn redoes kNN on CPU)."""
    def __init__(self, limit=40):
        self.limit, self._d, self._order = limit, {}, []
    def get(self, path):
        if path in self._d:
            return self._d[path]
        pts, lbl = load_pointcloud(path)
        if lbl is None:
            lbl = np.zeros(len(pts), np.int64)          # unlabeled -> all class 0
        lbl = np.clip(lbl, 0, CONFIG["num_classes"] - 1).astype(np.int64)
        feat = build_features(pts, CONFIG["normal_k"])
        entry = dict(raw=pts, feat=feat, lbl=lbl)
        self._d[path] = entry; self._order.append(path)
        if len(self._order) > self.limit:
            self._d.pop(self._order.pop(0), None)
        return entry

CACHE = CloudCache()
# FIX: cache must hold all train+val files, else per-epoch recomputation of normals
CACHE.limit = len(SPLIT["train"]) + len(SPLIT["val"]) + len(SPLIT["test"]) + 10
log.info("cache limit set to %d (covers every split file)", CACHE.limit)

# ---- disk-persistent feature cache: compute once, reuse across sessions ----
import pickle
_cache_file = CONFIG["feature_cache_file"]
if os.path.exists(_cache_file):
    try:
        with open(_cache_file, "rb") as fh:
            CACHE._d = pickle.load(fh)
        CACHE._order = list(CACHE._d.keys())
        log.info("loaded %d files' features from %s (skipping recomputation)",
                 len(CACHE._d), _cache_file)
    except Exception as e:
        log.warning("feature cache unreadable (%s) -> recomputing", e)
missing = [f for f in SPLIT["train"] + SPLIT["val"] if f not in CACHE._d]
if missing:
    log.info("pre-computing features for %d file(s) (one-time cost)...", len(missing))
    for f in tqdm(missing, desc="features"):
        CACHE.get(f)
    try:
        with open(_cache_file, "wb") as fh:
            pickle.dump(CACHE._d, fh)
        log.info("saved feature cache -> %s", _cache_file)
    except Exception as e:
        log.warning("could not save feature cache (%s)", e)

## Dataset & PTv2 Model

Same sphere-crop chunk sampling and PTv2 architecture as the main benchmark
(grouped vector attention + grid pooling), just fed 7-channel input and with no
Open3D anywhere in the pipeline.

In [ ]:
# ================================================================ DATASET
class ChunkDataset(Dataset):
    def __init__(self, files, augment=True):
        self.files, self.augment = list(files), augment
        self.cpc = CONFIG["chunks_per_cloud"]
        self.N = CONFIG["num_points"]
    def __len__(self): return len(self.files) * self.cpc
    def __getitem__(self, i):
        e = CACHE.get(self.files[i // self.cpc])
        feat, lbl = e["feat"], e["lbl"]
        n = len(feat)
        if n <= self.N:
            idx = np.random.choice(n, self.N, replace=True)
        else:
            seed = np.random.randint(n)
            d2 = ((feat[:, :3] - feat[seed, :3]) ** 2).sum(1)
            idx = np.argpartition(d2, self.N - 1)[:self.N]
        x, y = feat[idx].copy(), lbl[idx].copy()
        if self.augment:
            th = np.random.uniform(0, 2 * np.pi); c, s = np.cos(th), np.sin(th)
            R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], np.float32)
            x[:, :3] = x[:, :3] @ R.T * np.random.uniform(0.9, 1.1) + \
                np.random.normal(0, 0.002, (len(x), 3)).astype(np.float32)
            x[:, 4:7] = x[:, 4:7] @ R.T                  # rotate normals too
        return torch.from_numpy(x.T), torch.from_numpy(y)   # (7,N), (N,)

def make_loader(files, shuffle, batch=None):
    return DataLoader(ChunkDataset(files, augment=shuffle),
                      batch_size=batch or CONFIG["batch_size"],
                      shuffle=shuffle, drop_last=shuffle, num_workers=0,
                      pin_memory=(DEVICE.type == "cuda"))

def class_weights(files, sample=30):
    counts = np.zeros(CONFIG["num_classes"])
    for f in files[:sample]:
        counts += np.bincount(CACHE.get(f)["lbl"], minlength=CONFIG["num_classes"])
    w = 1.0 / np.clip(counts / counts.sum(), 1e-6, None)
    w = w / w.sum() * CONFIG["num_classes"]
    return torch.tensor(w, dtype=torch.float32)

CLASS_W = class_weights(SPLIT["train"])
log.info("class weights: %s", CLASS_W.numpy().round(3))

In [ ]:
# ================================================================ POINTTRANSFORMER V2
IN_CH = 7   # xyz + height + normal

class GVA(nn.Module):
    """Grouped vector attention with multiplier+bias position encoding."""
    def __init__(self, ch, groups=6, k=16):
        super().__init__()
        assert ch % groups == 0
        self.k, self.g, self.gc = k, groups, ch // groups
        self.q, self.kk, self.v = (nn.Linear(ch, ch) for _ in range(3))
        self.pm = nn.Sequential(nn.Linear(3, ch), nn.ReLU(), nn.Linear(ch, ch))
        self.pb = nn.Sequential(nn.Linear(3, ch), nn.ReLU(), nn.Linear(ch, ch))
        self.w = nn.Sequential(nn.Linear(ch, ch), nn.ReLU(), nn.Linear(ch, groups))

    def forward(self, x, pos, batch):
        e = tc_knn(pos, pos, self.k, batch, batch)
        c, nb = e[0], e[1]
        dp = pos[c] - pos[nb]
        pb = self.pb(dp)
        rel = (self.q(x)[c] - self.kk(x)[nb]) * self.pm(dp) + pb
        w = scatter_softmax(self.w(rel), c, dim=0)
        vg = (self.v(x)[nb] + pb).view(-1, self.g, self.gc)
        out = scatter_add(vg * w.unsqueeze(-1), c, dim=0, dim_size=x.size(0))
        return out.view(-1, self.g * self.gc)

class PTv2Block(nn.Module):
    def __init__(self, ch, groups=6, k=16):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(ch), nn.LayerNorm(ch)
        self.attn = GVA(ch, groups, k)
        self.mlp = nn.Sequential(nn.Linear(ch, ch * 2), nn.ReLU(), nn.Linear(ch * 2, ch))
    def forward(self, x, pos, batch):
        x = x + self.attn(self.n1(x), pos, batch)
        return x + self.mlp(self.n2(x))

class GridPool(nn.Module):
    def __init__(self, i, o, grid):
        super().__init__()
        self.grid = grid
        self.proj = nn.Sequential(nn.Linear(i, o), nn.ReLU())
    def forward(self, x, pos, batch):
        vox = torch.floor(pos / self.grid).long()
        key = torch.cat([batch.unsqueeze(1), vox], 1)
        _, cl = torch.unique(key, dim=0, return_inverse=True)
        xp, _ = scatter_max(self.proj(x), cl, dim=0)
        return xp, scatter_mean(pos, cl, dim=0), scatter_max(batch, cl, dim=0)[0], cl

class PTv2Seg(nn.Module):
    def __init__(self, num_classes, k=16, dims=(48, 96, 192), g=6):
        super().__init__()
        d = dims
        self.embed = nn.Sequential(nn.Linear(IN_CH, d[0]), nn.ReLU(), nn.Linear(d[0], d[0]))
        self.e1 = PTv2Block(d[0], g, k)
        self.p1 = GridPool(d[0], d[1], 0.08)
        self.e2 = PTv2Block(d[1], g, k)
        self.p2 = GridPool(d[1], d[2], 0.16)
        self.e3 = PTv2Block(d[2], g, k)
        self.u2 = nn.Sequential(nn.Linear(d[2] + d[1], d[1]), nn.ReLU())
        self.d2 = PTv2Block(d[1], g, k)
        self.u1 = nn.Sequential(nn.Linear(d[1] + d[0], d[0]), nn.ReLU())
        self.d1 = PTv2Block(d[0], g, k)
        self.head = nn.Sequential(nn.LayerNorm(d[0]), nn.Linear(d[0], 128), nn.ReLU(),
                                  nn.Dropout(0.4), nn.Linear(128, num_classes))

    def forward(self, x):
        B, C, N = x.shape
        flat = x.permute(0, 2, 1).reshape(-1, C).contiguous()
        p0 = flat[:, :3].contiguous()
        b0 = torch.arange(B, device=x.device).repeat_interleave(N)
        h0 = self.e1(self.embed(flat), p0, b0)
        h1, p1, b1, c1 = self.p1(h0, p0, b0)
        h1 = self.e2(h1, p1, b1)
        h2, p2, b2, c2 = self.p2(h1, p1, b1)
        h2 = self.e3(h2, p2, b2)
        u1 = self.d2(self.u2(torch.cat([h1, h2[c2]], 1)), p1, b1)
        u0 = self.d1(self.u1(torch.cat([h0, u1[c1]], 1)), p0, b0)
        return self.head(u0).view(B, N, -1).permute(0, 2, 1)   # (B,C,N)

## Metrics, Training, Validation, Testing

Standard segmentation metrics (accuracy, per-class IoU, mIoU, Dice, confusion
matrix), an OOM-safe training loop (dynamic batch halving + AMP + gradient
accumulation), early stopping on validation mIoU, and a held-out **test** split
evaluated once at the end.

In [ ]:
# ================================================================ METRICS
def seg_metrics(conf):
    conf = conf.astype(np.float64); k = conf.shape[0]
    out = {"accuracy": float(np.diag(conf).sum() / max(conf.sum(), 1))}
    ious, dices = [], []
    for c in range(k):
        tp = conf[c, c]; fp = conf[:, c].sum() - tp; fn = conf[c, :].sum() - tp
        iou = tp / (tp + fp + fn) if tp + fp + fn > 0 else float("nan")
        dice = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else float("nan")
        out[f"iou_{c}"] = float(iou); out[f"dice_{c}"] = float(dice)
        ious.append(iou); dices.append(dice)
    out["miou"] = float(np.nanmean(ious)); out["dice"] = float(np.nanmean(dices))
    return out

def confusion(gt, pred, k):
    return np.bincount(gt * k + pred, minlength=k * k).reshape(k, k)

In [ ]:
# ================================================================ TRAINING (OOM-safe)
def train_ptv2():
    model = PTv2Seg(CONFIG["num_classes"], k=CONFIG["k_neighbors"]).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=CLASS_W.to(DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                            weight_decay=CONFIG["weight_decay"])
    amp = CONFIG["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=amp)
    batch = CONFIG["batch_size"]; accum = max(1, CONFIG["grad_accum"])

    class _RetryEpoch(Exception): pass

    def run_epoch(loader, training):
        nonlocal batch
        model.train(training)
        tot, conf = 0.0, np.zeros((CONFIG["num_classes"],) * 2, np.int64)
        ctx = torch.enable_grad() if training else torch.no_grad()
        with ctx:
            for step, (x, y) in enumerate(loader):
                try:
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    with torch.autocast(DEVICE.type, enabled=amp):
                        logits = model(x)
                        loss = crit(logits, y) / accum
                    if training:
                        scaler.scale(loss).backward()
                        if (step + 1) % accum == 0:
                            scaler.step(opt); scaler.update()
                            opt.zero_grad(set_to_none=True)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if batch > 1:
                        batch = max(1, batch // 2)
                        log.warning("CUDA OOM -> retrying with batch=%d", batch)
                    else:
                        log.warning("CUDA OOM at batch=1 -> falling back to CPU")
                        globals()["DEVICE"] = torch.device("cpu")
                        model.to(DEVICE); crit.to(DEVICE)
                    raise _RetryEpoch()
                tot += loss.item() * accum * x.size(0)
                conf += confusion(y.cpu().numpy().ravel(),
                                  logits.argmax(1).detach().cpu().numpy().ravel(),
                                  CONFIG["num_classes"])
        return tot / max(len(loader.dataset), 1), seg_metrics(conf)

    best_state, best_miou, bad, history = None, -1.0, 0, []
    ep = 1
    while ep <= CONFIG["epochs"]:
        try:
            tr_loss, tr_m = run_epoch(make_loader(SPLIT["train"], True, batch), True)
            va_loss, va_m = run_epoch(make_loader(SPLIT["val"], False, batch), False)
        except _RetryEpoch:
            continue
        history.append({"epoch": ep, "train_loss": tr_loss, "val_loss": va_loss,
                        "val_miou": va_m["miou"], "val_dice": va_m["dice"],
                        "val_acc": va_m["accuracy"]})
        star = ""
        if va_m["miou"] > best_miou + 1e-4:
            best_miou, bad = va_m["miou"], 0
            best_state = copy.deepcopy(model.state_dict()); star = "  *best"
        else:
            bad += 1
        log.info("ep %03d | train_loss %.4f | val_loss %.4f | val_mIoU %.4f%s",
                 ep, tr_loss, va_loss, va_m["miou"], star)
        if bad >= CONFIG["patience"]:
            log.info("early stop (no val mIoU improvement for %d epochs)",
                     CONFIG["patience"])
            break
        ep += 1
    if best_state:
        model.load_state_dict(best_state)
    torch.save({"model_state": model.state_dict(), "config": CONFIG,
               "best_val_miou": best_miou, "history": history},
              CONFIG["checkpoint_path"])
    log.info("saved best model -> %s (val mIoU %.4f)",
             CONFIG["checkpoint_path"], best_miou)
    return model, history

MODEL, HISTORY = train_ptv2()

In [ ]:
# ================================================================ FULL-CLOUD PREDICTION
@torch.no_grad()
def predict_full_cloud(model, feat, chunk=None):
    """Label every point via shuffled fixed-size chunks -> batched forward -> scatter."""
    model.eval()
    N = CONFIG["num_points"]; B = chunk or CONFIG["batch_size"]
    n = len(feat)
    perm = np.random.RandomState(0).permutation(n)
    pad = (N - n % N) % N
    if pad:
        perm = np.concatenate([perm, perm[:pad]])
    chunks = perm.reshape(-1, N)
    out = np.zeros(n, np.int64)
    i = 0
    while i < len(chunks):
        cid = chunks[i:i + B]
        x = torch.from_numpy(feat[cid].transpose(0, 2, 1)).to(DEVICE)
        try:
            pred = model(x).argmax(1).cpu().numpy()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if B > 1:
                B = max(1, B // 2); continue
            model.to("cpu"); x = x.cpu(); pred = model(x).argmax(1).numpy()
            model.to(DEVICE)
        out[cid.ravel()] = pred.ravel()
        i += B
    return out

In [ ]:
# ================================================================ TESTING (held-out split)
def evaluate_files(model, files, label=""):
    conf = np.zeros((CONFIG["num_classes"],) * 2, np.int64)
    for f in tqdm(files, desc=f"testing [{label}]", leave=False):
        e = CACHE.get(f)
        pred = predict_full_cloud(model, e["feat"])
        conf += confusion(e["lbl"], pred, CONFIG["num_classes"])
    metrics = seg_metrics(conf)
    print(f"\n=== TEST RESULTS [{label}] ({len(files)} files) ===")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"mIoU     : {metrics['miou']:.4f}")
    print(f"Dice     : {metrics['dice']:.4f}")
    for c in range(CONFIG["num_classes"]):
        print(f"  class {c}: IoU {metrics[f'iou_{c}']:.4f} | Dice {metrics[f'dice_{c}']:.4f}")
    print("Confusion matrix (rows=GT, cols=predicted):")
    print(conf)
    return metrics, conf

TEST_METRICS, TEST_CONF = evaluate_files(MODEL, SPLIT["test"], label="held-out test")

# ---- learning curves ----
import matplotlib.pyplot as plt
h = HISTORY
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot([x["epoch"] for x in h], [x["train_loss"] for x in h], label="train")
plt.plot([x["epoch"] for x in h], [x["val_loss"] for x in h], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Loss")
plt.subplot(1, 2, 2)
plt.plot([x["epoch"] for x in h], [x["val_miou"] for x in h])
plt.xlabel("epoch"); plt.ylabel("val mIoU"); plt.title("Validation mIoU")
plt.tight_layout(); plt.show()

## Final Inference on `data/test` (no metrics if unlabeled)

In [ ]:
# ================================================================ FINAL INFERENCE
if TEST_FILES:
    print(f"Running inference on {len(TEST_FILES)} file(s) in '{CONFIG['test_dir']}'")
    for f in TEST_FILES:
        pts, lbl = load_pointcloud(f)
        feat = build_features(pts, CONFIG["normal_k"])
        pred = predict_full_cloud(MODEL, feat)
        n1 = int((pred == 1).sum())
        line = f"{os.path.basename(f)} | n_points={len(pts)} | predicted class-1={n1}"
        if lbl is not None and len(np.unique(lbl)) > 1:
            m, _ = evaluate_files(MODEL, [f], label=os.path.basename(f))
            line += f" | mIoU={m['miou']:.4f}"
        print(line)
else:
    log.info("data/test is empty -> nothing to run inference on")